In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import os
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases/Glass_Identification")

In [52]:
df = pd.read_csv("Glass.csv")
df

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,building_windows_float_processed
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,building_windows_float_processed
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,building_windows_float_processed
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,building_windows_float_processed
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,building_windows_float_processed
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,headlamps
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,headlamps
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,headlamps
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,headlamps


In [53]:
le = LabelEncoder()
df["Type"] = le.fit_transform(df["Type"])

In [54]:
X,y = df.drop('Type',axis=1),df['Type']

In [44]:
# IF = IsolationForest(random_state=26,contamination=0.05)
IF = IsolationForest(random_state=26)
IF.fit(X)
pred_outliers = IF.predict(X)

In [45]:
pd.crosstab(colnames=['Outlier or Not'],rownames=['Type of Glass'],columns=pred_outliers,index=y,margins=True)

Outlier or Not,-1,1,All
Type of Glass,,,
building_windows_float_processed,0,70,70
building_windows_non_float_processed,8,68,76
containers,5,8,13
headlamps,8,21,29
tableware,1,8,9
vehicle_windows_float_processed,1,16,17
All,23,191,214


In [47]:
print(f"Percentage of outliers: {(pred_outliers<0).mean():.6%}")

Percentage of outliers: 10.747664%


In [48]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [57]:
in_glass = df[pred_outliers != -1]
kfold = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 26)
X,y = in_glass.drop('Type',axis=1),in_glass['Type']
params = {"learning_rate" : np.linspace(0.001, 0.9, 30), "n_estimators" : [10, 25, 50, 75, 100], "max_depth" : [2, 3, 4, 5, 6]}
gbm = XGBClassifier(random_state=26, n_jobs = -1)
gcv = GridSearchCV(gbm, param_grid = params, cv = kfold, scoring = 'neg_log_loss',verbose=3)
gcv.fit(X, y)

Fitting 5 folds for each of 750 candidates, totalling 3750 fits
[CV 1/5] END learning_rate=0.001, max_depth=2, n_estimators=10;, score=-1.446 total time=   0.0s
[CV 2/5] END learning_rate=0.001, max_depth=2, n_estimators=10;, score=-1.423 total time=   0.0s
[CV 3/5] END learning_rate=0.001, max_depth=2, n_estimators=10;, score=-1.423 total time=   0.0s
[CV 4/5] END learning_rate=0.001, max_depth=2, n_estimators=10;, score=-1.464 total time=   0.0s
[CV 5/5] END learning_rate=0.001, max_depth=2, n_estimators=10;, score=-1.478 total time=   0.0s
[CV 1/5] END learning_rate=0.001, max_depth=2, n_estimators=25;, score=-1.437 total time=   0.0s
[CV 2/5] END learning_rate=0.001, max_depth=2, n_estimators=25;, score=-1.410 total time=   0.0s
[CV 3/5] END learning_rate=0.001, max_depth=2, n_estimators=25;, score=-1.410 total time=   0.0s
[CV 4/5] END learning_rate=0.001, max_depth=2, n_estimators=25;, score=-1.454 total time=   0.0s
[CV 5/5] END learning_rate=0.001, max_depth=2, n_estimators=25;

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'learning_rate': array([0.001,...0.869, 0.9 ]), 'max_depth': [2, 3, ...], 'n_estimators': [10, 25, ...]}"
,scoring,'neg_log_loss'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'multi:softprob'


In [59]:
gcv.best_params_, gcv.best_score_

({'learning_rate': np.float64(0.6829999999999999),
  'max_depth': 6,
  'n_estimators': 10},
 np.float64(-0.6708996359025863))